# Recurrent Neural Network Example

Xây dựng mạng RNN với PyTorch

## Tổng quan về RNN

<img src="http://colah.github.io/posts/2015-08-Understanding-LSTMs/img/RNN-unrolled.png" alt="nn" style="width: 600px;"/>

Tài liệu tham khảo:
- [Long Short Term Memory](http://deeplearning.cs.cmu.edu/pdfs/Hochreiter97_lstm.pdf), Sepp Hochreiter & Jurgen Schmidhuber, Neural Computation 9(8): 1735-1780, 1997.

## Tổng quan về bộ dữ liệu MNIST

Ví dụ này sử dụng bộ dữ liệu về chữ số viết tay MNIST. Bộ dữ liệu chữa 60k mẫu cho huấn luyện và 10k mẫu cho kiểm thử.

![MNIST Dataset](https://i1.wp.com/varianceexplained.org/images/mnist.png?w=456)


Để phân loại hình ảnh sử dụng RNN, chúng ta sẽ coi mỗi hàng là 1 chuỗi pixels. Bởi vì kích thước ảnh là 28*28px, ta sẽ sử lý 28 chuỗi của 28 timesteps cho tất cả các sample.

In [1]:
from __future__ import absolute_import, division, print_function

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.autograd import Variable
import numpy as np

In [2]:
from torchvision.datasets import MNIST
from torchvision import transforms

# Load MNIST using torchvision
transform = transforms.ToTensor()
mnist_train = MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = MNIST(root='./data', train=False, download=True, transform=transform)

# Convert to numpy for processing
x_train = mnist_train.data.numpy().astype(np.float32) / 255.0  # [0, 1]
y_train = mnist_train.targets.numpy()
x_test = mnist_test.data.numpy().astype(np.float32) / 255.0
y_test = mnist_test.targets.numpy()

# Reshape: (N, 28, 28) -> (N, 28, 28) for sequence processing
# We treat 28 rows as 28 timesteps of 28-dim input
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train).type(torch.LongTensor)
y_test = torch.from_numpy(y_test).type(torch.LongTensor)

print(f'x_train shape: {x_train.shape}')  # (60000, 28, 28)
print(f'y_train shape: {y_train.shape}')  # (60000,)


100%|██████████| 9.91M/9.91M [00:00<00:00, 16.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 484kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.7MB/s]


x_train shape: torch.Size([60000, 28, 28])
y_train shape: torch.Size([60000])


In [3]:
# x_train.shape

In [4]:
# Create DataLoader for batch processing
batch_size = 16

class MNISTSequenceDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

train_dataset = MNISTSequenceDataset(x_train, y_train)
test_dataset = MNISTSequenceDataset(x_test, y_test)

trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [5]:
class RNNModel(nn.Module):
    """RNN model for MNIST classification
    Treats 28x28 image as sequence of 28 timesteps, each with 28 features
    Input: (batch, seq_len=28, input_size=28)
    Output: (batch, num_classes=10)
    """
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(RNNModel, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        
        # RNN layer: processes sequences
        # input_dim=28 (pixel values per row)
        # hidden_dim=hidden units
        # layer_dim=number of RNN layers
        # batch_first=True: input shape is (batch, seq, features)
        self.rnn = nn.RNN(input_dim, hidden_dim, layer_dim, 
                          batch_first=True, nonlinearity='relu', 
                          bidirectional=False)
        
        # Fully connected output layer
        # Takes final hidden state and outputs class logits
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        # x shape: (batch_size, seq_len=28, input_size=28)
        
        # Initialize hidden state with zeros
        # Shape: (num_layers, batch_size, hidden_dim)
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim)
            
        # Forward pass through RNN
        # out shape: (batch, seq_len, hidden_dim)
        # hn shape: (num_layers, batch, hidden_dim)
        out, hn = self.rnn(x, h0)
        
        # Take output from last timestep and pass through FC layer
        # out[:, -1, :] shape: (batch, hidden_dim)
        out = self.fc(out[:, -1, :])
        return out


In [6]:
input_dim = 28    # Each row has 28 features (pixels)
hidden_dim = 100  # Number of hidden units
layer_dim = 1     # Number of RNN layers
output_dim = 10   # 10 digit classes

# Create RNN model
model = RNNModel(input_dim, hidden_dim, layer_dim, output_dim)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)


In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

num_epochs = 2

for epoch in range(num_epochs):
    running_loss = 0.0
    
    for i, (inputs, labels) in enumerate(trainloader):
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Print statistics
        running_loss += loss.item()
        if (i + 1) % 1000 == 0:
            print(f'[Epoch {epoch + 1}, Batch {i + 1}] loss: {running_loss / 1000:.3f}')
            running_loss = 0.0

print('Finished Training')


[Epoch 1, Batch 1000] loss: 2.300
[Epoch 1, Batch 2000] loss: 2.288
[Epoch 1, Batch 3000] loss: 1.757
[Epoch 2, Batch 1000] loss: 0.756
[Epoch 2, Batch 2000] loss: 0.596
[Epoch 2, Batch 3000] loss: 0.496
Finished Training


In [8]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy of the network on 10000 test images: {accuracy:.2f}%')


Accuracy of the network on 10000 test images: 88.65%
